# 05 — Model Training

**Objective:** Train baseline and candidate forecasting models on `train.csv`, tune the strongest candidate against `val.csv`, and save all trained models (PRD Milestone 4 / FR-ML-001..003, ML-MODEL-001..004, ML-OPT-001).

**Models trained:**
- **Linear Regression** — mandatory baseline (ML-MODEL-001).
- **Random Forest** — mandatory (ML-MODEL-002), then hyperparameter-tuned.
- **Gradient Boosting** — optional (ML-MODEL-003), included at zero extra dependency cost since it ships with scikit-learn.

**Feature set:** Group A columns only (`data/metadata/FEATURE_DOCUMENTATION.md`) via `build_model_matrix()` — pollutant columns and `Year` are excluded (leakage risk and extrapolation risk, respectively; see that document and `03_exploratory_data_analysis.ipynb`).

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np

from src.feature_engineering.model_matrix import build_model_matrix
from src.models.model_trainer import train_model, get_feature_importance, MODEL_REGISTRY
from src.models.hyperparameter_tuning import tune_with_grid_search
from src.models.model_io import save_model
from src.evaluation.metrics import calculate_regression_metrics
from config.paths import PROCESSED_DATA_DIR, TRAINED_MODELS_DIR
from config.constants import DEFAULT_RANDOM_SEED

train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv", parse_dates=["Date"])
val_df = pd.read_csv(PROCESSED_DATA_DIR / "val.csv", parse_dates=["Date"])

X_train, y_train, feature_names = build_model_matrix(train_df)
X_val, y_val, _ = build_model_matrix(val_df)

print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Val:   X={X_val.shape}, y={y_val.shape}")
print(f"Features ({len(feature_names)}): {feature_names}")

2026-09-06 17:06:26,540 | INFO | src.feature_engineering.model_matrix | build_model_matrix: 1470 rows, 24 features.
2026-09-06 17:06:26,543 | INFO | src.feature_engineering.model_matrix | build_model_matrix: 315 rows, 24 features.


Train: X=(1470, 24), y=(1470,)
Val:   X=(315, 24), y=(315,)
Features (24): ['Month', 'Quarter', 'Week', 'Day', 'DayOfWeek', 'IsWeekend', 'DayOfYear', 'Lag_1', 'Lag_3', 'Lag_7', 'Lag_14', 'Lag_30', 'Rolling_Mean_7', 'Rolling_Mean_14', 'Rolling_Mean_30', 'Rolling_Median_7', 'Rolling_Median_14', 'Rolling_Median_30', 'Rolling_Std_7', 'Rolling_Std_14', 'Rolling_Std_30', 'Season_Post-Monsoon', 'Season_Summer', 'Season_Winter']


## Baseline — Linear Regression (ML-MODEL-001)

In [2]:
lr_model = train_model("linear_regression", X_train, y_train)
lr_val_preds = lr_model.predict(X_val)
lr_val_metrics = calculate_regression_metrics(y_val, lr_val_preds)
print(f"Linear Regression -- validation set: {lr_val_metrics.summary()}")

2026-09-06 17:06:26,559 | INFO | src.models.model_trainer | train_model: trained 'linear_regression' on 1470 rows, 24 features.
2026-09-06 17:06:26,563 | INFO | src.evaluation.metrics | calculate_regression_metrics: MAE=25.91  MSE=1273.52  RMSE=35.69  R2=0.9012


Linear Regression -- validation set: MAE=25.91  MSE=1273.52  RMSE=35.69  R2=0.9012


**Interpretation:** this is the benchmark every other model must beat to justify its added complexity (Handbook D.39, Baseline Model Policy).

## Random Forest — untuned (ML-MODEL-002)

In [3]:
rf_model = train_model("random_forest", X_train, y_train)
rf_val_preds = rf_model.predict(X_val)
rf_val_metrics = calculate_regression_metrics(y_val, rf_val_preds)
print(f"Random Forest (default params) -- validation set: {rf_val_metrics.summary()}")

2026-09-06 17:06:27,187 | INFO | src.models.model_trainer | train_model: trained 'random_forest' on 1470 rows, 24 features.
2026-09-06 17:06:27,239 | INFO | src.evaluation.metrics | calculate_regression_metrics: MAE=26.90  MSE=1369.71  RMSE=37.01  R2=0.8938


Random Forest (default params) -- validation set: MAE=26.90  MSE=1369.71  RMSE=37.01  R2=0.8938


## Random Forest — hyperparameter tuning (ML-OPT-001)\n\nUsing `TimeSeriesSplit` cross-validation, never plain shuffled K-Fold (Handbook D.38).

In [4]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [5, 10, 15, None],
    "min_samples_leaf": [1, 5, 10],
}

tuned_rf, best_params, best_cv_mae = tune_with_grid_search(
    rf_model.__class__(random_state=DEFAULT_RANDOM_SEED, n_jobs=-1),
    param_grid, X_train, y_train, n_splits=5,
)
print(f"Best params: {best_params}")
print(f"Best CV MAE (TimeSeriesSplit, on train set only): {best_cv_mae:.2f}")

tuned_rf_val_preds = tuned_rf.predict(X_val)
tuned_rf_val_metrics = calculate_regression_metrics(y_val, tuned_rf_val_preds)
print(f"\nTuned Random Forest -- validation set: {tuned_rf_val_metrics.summary()}")

2026-09-06 17:07:14,391 | INFO | src.models.hyperparameter_tuning | tune_with_grid_search: best_params={'max_depth': 10, 'min_samples_leaf': 10, 'n_estimators': 100}, best_score=35.812
2026-09-06 17:07:14,468 | INFO | src.evaluation.metrics | calculate_regression_metrics: MAE=26.72  MSE=1379.87  RMSE=37.15  R2=0.8930


Best params: {'max_depth': 10, 'min_samples_leaf': 10, 'n_estimators': 100}
Best CV MAE (TimeSeriesSplit, on train set only): 35.81

Tuned Random Forest -- validation set: MAE=26.72  MSE=1379.87  RMSE=37.15  R2=0.8930


## Gradient Boosting — optional model (ML-MODEL-003)

In [5]:
gb_model = train_model("gradient_boosting", X_train, y_train)
gb_val_preds = gb_model.predict(X_val)
gb_val_metrics = calculate_regression_metrics(y_val, gb_val_preds)
print(f"Gradient Boosting -- validation set: {gb_val_metrics.summary()}")

2026-09-06 17:07:15,076 | INFO | src.models.model_trainer | train_model: trained 'gradient_boosting' on 1470 rows, 24 features.
2026-09-06 17:07:15,079 | INFO | src.evaluation.metrics | calculate_regression_metrics: MAE=27.22  MSE=1396.09  RMSE=37.36  R2=0.8917


Gradient Boosting -- validation set: MAE=27.22  MSE=1396.09  RMSE=37.36  R2=0.8917


## Validation-set comparison (before touching the test set)

In [6]:
comparison = pd.DataFrame([
    {"Model": "Linear Regression (baseline)", **lr_val_metrics.to_dict()},
    {"Model": "Random Forest (default)", **rf_val_metrics.to_dict()},
    {"Model": "Random Forest (tuned)", **tuned_rf_val_metrics.to_dict()},
    {"Model": "Gradient Boosting", **gb_val_metrics.to_dict()},
]).sort_values("MAE")
print(comparison.to_string(index=False))

                       Model       MAE         MSE      RMSE       R2
Linear Regression (baseline) 25.914604 1273.521733 35.686436 0.901214
       Random Forest (tuned) 26.722463 1379.874123 37.146657 0.892965
     Random Forest (default) 26.901571 1369.710918 37.009606 0.893753
           Gradient Boosting 27.215975 1396.086606 37.364242 0.891707


**Interpretation:** ranked by MAE. Selection made entirely on the validation set — the test set stays untouched until `06_model_evaluation.ipynb`.

## Feature importance (best model)

In [7]:
best_model_row = comparison.iloc[0]
print(f"Best model by validation MAE: {best_model_row['Model']}")

models_by_name = {
    "Linear Regression (baseline)": lr_model,
    "Random Forest (default)": rf_model,
    "Random Forest (tuned)": tuned_rf,
    "Gradient Boosting": gb_model,
}
best_model = models_by_name[best_model_row["Model"]]

importances = get_feature_importance(best_model, feature_names)
top_10 = dict(list(importances.items())[:10])
for feat, score in top_10.items():
    print(f"  {feat:<20} {score:.4f}")

Best model by validation MAE: Linear Regression (baseline)
  Month                66.0230
  Season_Winter        25.7409
  Season_Post-Monsoon  24.5082
  Season_Summer        10.7887
  Quarter              4.4854
  IsWeekend            4.3084
  DayOfYear            2.1555
  Day                  2.1279
  Rolling_Mean_7       1.1651
  DayOfWeek            1.1277


## Save all trained models (ML-REG-001)

In [8]:
models_to_save = {
    "linear_regression_v1": (lr_model, lr_val_metrics),
    "random_forest_v1": (rf_model, rf_val_metrics),
    "random_forest_tuned_v1": (tuned_rf, tuned_rf_val_metrics),
    "gradient_boosting_v1": (gb_model, gb_val_metrics),
}

for name, (model, val_metrics) in models_to_save.items():
    save_model(
        model,
        TRAINED_MODELS_DIR / f"{name}.joblib",
        metadata={
            "algorithm": type(model).__name__,
            "feature_names": feature_names,
            "training_dataset": "data/processed/train.csv",
            "training_rows": len(X_train),
            "validation_metrics": val_metrics.to_dict(),
            "random_seed": DEFAULT_RANDOM_SEED,
        },
    )

print(f"Saved {len(models_to_save)} models to {TRAINED_MODELS_DIR}")
import os
for f in sorted(os.listdir(TRAINED_MODELS_DIR)):
    print(f"  {f}")

2026-09-06 17:07:15,122 | INFO | src.models.model_io | save_model: saved 'D:\AQI_Forecast_Platform\models\trained\linear_regression_v1.joblib' with metadata 'D:\AQI_Forecast_Platform\models\trained\linear_regression_v1.metadata.json'.
2026-09-06 17:07:15,183 | INFO | src.models.model_io | save_model: saved 'D:\AQI_Forecast_Platform\models\trained\random_forest_v1.joblib' with metadata 'D:\AQI_Forecast_Platform\models\trained\random_forest_v1.metadata.json'.
2026-09-06 17:07:15,204 | INFO | src.models.model_io | save_model: saved 'D:\AQI_Forecast_Platform\models\trained\random_forest_tuned_v1.joblib' with metadata 'D:\AQI_Forecast_Platform\models\trained\random_forest_tuned_v1.metadata.json'.
2026-09-06 17:07:15,212 | INFO | src.models.model_io | save_model: saved 'D:\AQI_Forecast_Platform\models\trained\gradient_boosting_v1.joblib' with metadata 'D:\AQI_Forecast_Platform\models\trained\gradient_boosting_v1.metadata.json'.


Saved 4 models to D:\AQI_Forecast_Platform\models\trained
  .gitkeep
  gradient_boosting_v1.joblib
  gradient_boosting_v1.metadata.json
  linear_regression_v1.joblib
  linear_regression_v1.metadata.json
  random_forest_tuned_v1.joblib
  random_forest_tuned_v1.metadata.json
  random_forest_v1.joblib
  random_forest_v1.metadata.json


## Conclusions

1. All 4 models trained successfully; ranked by validation MAE above.
2. Hyperparameter tuning used `TimeSeriesSplit` — no future-leakage into tuning.
3. All 4 models saved with full metadata sidecars.
4. The test set was never touched in this notebook.

## Next steps
→ `06_model_evaluation.ipynb`: final evaluation on the untouched test set, residual analysis, actual-vs-predicted plots, and forecast output generation.